# Google WAXAL ASR Challenge: Qwen3-ASR-1.7B Fine-Tuning

In [ ]:
!pip uninstall -y torchao
!apt-get update -y && apt-get install -y ffmpeg
!pip install -q -U transformers
!pip install -q peft datasets evaluate librosa jiwer accelerate trl

In [ ]:
import os
import torch
import librosa
from datasets import load_dataset, Audio
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
import warnings
warnings.filterwarnings('ignore')

In [ ]:
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

# Inizializzazione
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3ASRForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none"
)

model.enable_input_require_grads()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
DATASET_ID = "google/WaxalNLP"

def load_huggingface_dataset(language="lug", split="train"):
    ds = load_dataset(DATASET_ID, name=f"{language}_asr", split=split, streaming=True)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    return ds

train_dataset = load_huggingface_dataset(language="lug", split="train")

In [ ]:
def format_example(example):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": "dummy"},
                {"type": "text", "text": "Trascrivi questo audio."}
            ]
        },
        {
            "role": "assistant", 
            "content": example['transcription'] 
        }
    ]
    text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
    
    inputs = processor(text=text, audio=example["audio"]["array"], sampling_rate=16000)
    result = {k: v[0] for k, v in inputs.items()}
    result["labels"] = result["input_ids"].clone() if hasattr(result["input_ids"], "clone") else result["input_ids"].copy()
    return result

train_dataset = train_dataset.map(format_example)

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class MultimodalDataCollator:
    pad_token_id: int

    def __call__(self, features):
        batch = {}
        
        # Pad input_ids e attention_mask
        input_ids = [torch.tensor(f["input_ids"]) for f in features]
        batch["input_ids"] = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_id)
        
        if "attention_mask" in features[0]:
            attention_mask = [torch.tensor(f["attention_mask"]) for f in features]
            batch["attention_mask"] = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
            
        # Pad labels a -100 per ignorare il pad nel calcolo della loss
        if "labels" in features[0]:
            labels = [torch.tensor(f["labels"]) for f in features]
            batch["labels"] = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
            
        # Pad input_features (audio)
        if "input_features" in features[0]:
            features_list = [torch.tensor(f["input_features"]) for f in features]
            # Solitamente Qwen3 restituisce [seq_len, dim]
            if features_list[0].shape[0] < features_list[0].shape[1]:
                # E' [dim, seq_len], trasponiamo prima di paddare
                transposed = [feat.T for feat in features_list]
                padded = torch.nn.utils.rnn.pad_sequence(transposed, batch_first=True, padding_value=0.0)
                batch["input_features"] = padded.transpose(1, 2)
            else:
                batch["input_features"] = torch.nn.utils.rnn.pad_sequence(features_list, batch_first=True, padding_value=0.0)

        if "input_features_mask" in features[0]:
            mask_list = [torch.tensor(f["input_features_mask"]) for f in features]
            batch["input_features_mask"] = torch.nn.utils.rnn.pad_sequence(mask_list, batch_first=True, padding_value=0)

        return batch

data_collator = MultimodalDataCollator(pad_token_id=processor.tokenizer.pad_token_id)

training_args = TrainingArguments(
    output_dir="./qwen3_asr_lora_waxal",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    bf16=True,
    logging_steps=5,
    optim="adamw_torch",
    remove_unused_columns=True,
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    data_collator=data_collator
)

print("Avvio del training...")
trainer.train()